In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv('tweets_with_sentiment.csv')
conn = sqlite3.connect('tweets.db') # basic connect
df.to_sql('posts', conn, if_exists='replace', index=False) # creates a table called posts in tweets.db and dumps the entire df onto it
print('Loaded', len(df), 'rows into tweets.db') 

Loaded 10000 rows into tweets.db


In [ ]:
def query(sql): #query helper
    return pd.read_sql_query(sql, conn)

In [ ]:
query("""
    SELECT DATE(Timestamp) as date,  
           ROUND(AVG(compound_score), 3) as avg_sentiment,
           COUNT(*) as tweet_count
    FROM posts
    GROUP BY DATE(Timestamp)
    ORDER BY date
""")

# DATE(Timestamp) strips the time part and keeps only the date
# AVG(compound_score) averages all compound scores for that day
# ROUND( ,3) keeps it to 3 decimal places. GROUP BY DATE(Timestamp) is what collapses all tweets from the same day into one row

,date,avg_sentiment,tweet_count
0,2023-01-01,0.443,67
1,2023-01-02,0.402,85
2,2023-01-03,0.463,83
3,2023-01-04,0.556,74
4,2023-01-05,0.421,82
...,...,...,...
130,2023-05-11,0.479,86
131,2023-05-12,0.436,85
132,2023-05-13,0.505,62
133,2023-05-14,0.466,65


In [ ]:
query("""
    SELECT sentiment_label,
           COUNT(*) as count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM posts), 1) as percentage
    FROM posts
    GROUP BY sentiment_label
    ORDER BY count DESC
""")

#(SELECT COUNT(*) FROM posts) runs a second query inside the main one to get the total number of tweets 
# which you then use to calculate each label's percentage. 
# The 100.0 forces float division, without it SQLite does integer division and you'd get 0 for anything under 1%

,sentiment_label,count,percentage
0,positive,7934,79.3
1,negative,1682,16.8
2,neutral,384,3.8


In [ ]:
query("""
    SELECT Username,
           COUNT(*) as tweet_count,
           SUM(Likes) as total_likes,
           SUM(Retweets) as total_retweets,
           SUM(Likes + Retweets) as total_engagement
    FROM posts
    GROUP BY Username
    ORDER BY total_engagement DESC
    LIMIT 10
""")

#Likes + Retweets adds the two columns together per row, 
# then SUM() adds all those up per user. This is your engagement metric 
#  a simple but effective one for a portfolio project.
# strftime('%H', Timestamp) is SQLite's way of extracting the hour from a timestamp 
# it returns a string like "09". CAST(... AS INTEGER) converts it to a number so it 
# sorts correctly as 0, 1, 2... instead of "0", "1", "10".. string sorting would mess up the order

,Username,tweet_count,total_likes,total_retweets,total_engagement
0,pjohnson,6,351,362,713
1,awilliams,5,225,306,531
2,amiller,4,271,253,524
3,fsmith,4,170,301,471
4,nbrown,5,203,267,470
5,jessicawilliams,4,216,251,467
6,christopher64,3,190,261,451
7,udavis,3,231,200,431
8,nwhite,4,230,199,429
9,wmitchell,4,123,269,392


In [6]:
conn.close()